In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_v8_t_arch import RRDB_UNet_v8_t

In [ ]:
device="cuda"
pad = 8

In [ ]:
model = RRDB_UNet_v8_t(
    num_in_ch=6,
    num_out_ch=3,
    highway_channels_base=64,
    processing_channels_base=32,
    num_grow_ch_base = 16,
    encoder_blocks=2,
    decoder_blocks=4,
    ae_channel_multipliers = [1,3,9,18],
    body_rrdb_blocks = 4,
    body_dit_blocks = 12,
    t_hidden_dim=1024,
    inference=True,
    #memory_efficient_inference_device = "cuda"
)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total_params - trainable

print(f"Trainable: {trainable:,}")
print(f"Frozen:    {frozen:,}")

print(model)

In [ ]:
#img = cv2.imread("tests/data/lq_4/photo1.jpg")
#img = cv2.imread("tests/data/lq_4/photo2.jpg")
img = cv2.imread("tests/data/lq_4/baboon.png")
#img = cv2.imread("tests/data/lq_4/comic.png")
#img = cv2.imread("tests/data/gt/comic.png")
#img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img, (8050,8050))
#img = cv2.resize(img, (4020*1, 4020*1))
scale = 4
img = cv2.resize(img, (int(img.shape[1]*scale),int(img.shape[0]*scale)))
print(img.shape)
import matplotlib.pyplot as plt
from PIL import Image
Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
# Note: pytorch appears to use different gpu code if you exceed some resolution causing it to be very slow.
# compiling with fixed resolution makes it fast again, but it requires bucketing.
# see esrgan_arch_test2

In [ ]:
import os
latest_num = 0
for i in os.listdir("experiments/train_flow_v8/models/"):
    if i[-4:] == ".pth":
        try:
            num = int(i.split(".")[0].split("_")[-1])
            if num > latest_num:
                latest_num = num
        except:pass

path = "experiments/train_flow_v8/models/net_g_"+str(latest_num)+".pth"
path

In [ ]:
from realesrgan.real_srflow import RealSRFLOW
from realesrgan.real_srfield import RealSRFIELD

model.to("cuda")
model.eval()
srflow = RealSRFLOW(path, model, pad, device)
#srflow = RealSRFLOW(None, model, pad, device)

In [ ]:
out = srflow.enhance(img, 5)
Image.fromarray(cv2.cvtColor(out[-1], cv2.COLOR_BGR2RGB))

In [ ]:
import matplotlib.pyplot as plt

#for i in [out[0], out[-1]]:
for i in out:
    plt.imshow(cv2.cvtColor(i, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
exit()